🧪 Taller 004: Procesamiento y Modelo de ML con Apache Spark en Google Colab
Objetivo: Aprender a usar Apache Spark para procesar un conjunto de datos y aplicar un modelo de clasificación.

Paso 1: Configurar Spark en Colab

In [2]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.3.0/spark-3.3.0-bin-hadoop3.tgz
!tar xf spark-3.3.0-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.3.0-bin-hadoop3"

import findspark
findspark.init()

Paso 2: Inicializar Spark Session

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SparkML").getOrCreate()
print("Spark Session initialized successfully!")

Spark Session initialized successfully!


Paso 2.1 Importar librerias

In [8]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator


Paso 3: Cargar y mostrar el dataset

In [10]:
# Download the file locally
!wget https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv -O /tmp/pima-indians-diabetes.data.csv

# Load the downloaded file into a Spark DataFrame
df = spark.read.csv(
    "/tmp/pima-indians-diabetes.data.csv",
    inferSchema=True, header=False
)
df.show(5)

--2025-07-31 04:03:16--  https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 23278 (23K) [text/plain]
Saving to: ‘/tmp/pima-indians-diabetes.data.csv’

/tmp/pima-indians-d 100%[===================>]  22.73K  --.-KB/s    in 0.002s  

2025-07-31 04:03:16 (9.12 MB/s) - ‘/tmp/pima-indians-diabetes.data.csv’ saved [23278/23278]

+---+---+---+---+---+----+-----+---+---+
|_c0|_c1|_c2|_c3|_c4| _c5|  _c6|_c7|_c8|
+---+---+---+---+---+----+-----+---+---+
|  6|148| 72| 35|  0|33.6|0.627| 50|  1|
|  1| 85| 66| 29|  0|26.6|0.351| 31|  0|
|  8|183| 64|  0|  0|23.3|0.672| 32|  1|
|  1| 89| 66| 23| 94|28.1|0.167| 21|  0|
|  0|137| 40| 35|168|43.1|2.288| 33|  1|
+---+---+---+---+---+----+-

Paso 4: Preparar Features

In [11]:
from pyspark.ml.feature import VectorAssembler

# Assuming the last column is the label (index 8) and the rest are features (indices 0-7)
feature_cols = df.columns[:-1]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_assembled = assembler.transform(df)

df_assembled.show(5)

+---+---+---+---+---+----+-----+---+---+--------------------+
|_c0|_c1|_c2|_c3|_c4| _c5|  _c6|_c7|_c8|            features|
+---+---+---+---+---+----+-----+---+---+--------------------+
|  6|148| 72| 35|  0|33.6|0.627| 50|  1|[6.0,148.0,72.0,3...|
|  1| 85| 66| 29|  0|26.6|0.351| 31|  0|[1.0,85.0,66.0,29...|
|  8|183| 64|  0|  0|23.3|0.672| 32|  1|[8.0,183.0,64.0,0...|
|  1| 89| 66| 23| 94|28.1|0.167| 21|  0|[1.0,89.0,66.0,23...|
|  0|137| 40| 35|168|43.1|2.288| 33|  1|[0.0,137.0,40.0,3...|
+---+---+---+---+---+----+-----+---+---+--------------------+
only showing top 5 rows



Paso 5: Dividir los datos en conjuntos de entrenamiento y prueba

In [12]:
# Assuming the label column is the last column (_c8)
train_data, test_data = df_assembled.randomSplit([0.7, 0.3], seed=42)

print("Training Dataset Count:", train_data.count())
print("Testing Dataset Count:", test_data.count())

Training Dataset Count: 569
Testing Dataset Count: 199


Paso 6: Entrenar un Modelo de Clasificación

In [13]:
from pyspark.ml.classification import LogisticRegression

# Assuming the label column is _c8
lr = LogisticRegression(labelCol="_c8", featuresCol="features", maxIter=10, regParam=0.001)

# Train the model
lr_model = lr.fit(train_data)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


Paso 7: Evaluar el Modelo

In [14]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Make predictions on the test data
predictions = lr_model.transform(test_data)

# Evaluate the model
evaluator = BinaryClassificationEvaluator(labelCol="_c8", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)

print(f"Area Under ROC (AUC) on Test Data: {auc}")

Area Under ROC (AUC) on Test Data: 0.8511574074074074
